decompresse zip in data

In [1]:
import requests, zipfile, os # requetes HTTP zip fichier et dossiers
from tqdm import tqdm
#create directory if it does not exist
os.makedirs("data", exist_ok=True)
url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
zip_path = "data/annotations.zip"
#download annotations if not 
if not os.path.exists("data/annotations/captions_train2017.json"):
    #send HTTP do download archive
    r = requests.get(url, stream=True)
    total = int(r.headers.get("content-length", 0))
    with open(zip_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            bar.update(len(chunk))
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("data")

print(os.listdir("data/annotations"))

  0%|          | 0.00/253M [00:10<?, ?B/s]


KeyboardInterrupt: 

In [ ]:
from pycocotools.coco import COCO
import random, json
#chargemet des captions
coco = COCO("data/annotations/captions_train2017.json")
ids = sorted(coco.getImgIds())
print("Images disponibles :", len(ids))      # ~118 287


random.seed(42)#fixe the hasard
random.shuffle(ids)

kb_ids  = ids[:10000]
val_ids = ids[10000:10100]
#create splits if not 
os.makedirs("splits", exist_ok=True)
#save th img in json 
json.dump({"kb": kb_ids, "val": val_ids}, open("splits/train_splits.json", "w"))
print(len(kb_ids), len(val_ids))  
# Verify that KB and validation contain different images
assert set(kb_ids).isdisjoint(set(val_ids)) #done

print("Knowledge Base:", len(kb_ids))
print("Validation:", len(val_ids))
print("Total:", len(set(kb_ids) | set(val_ids)))          #done  # 10000 100

loading annotations into memory...


FileNotFoundError: [Errno 2] No such file or directory: 'data/annotations/captions_train2017.json'

download imgs

In [ ]:
from concurrent.futures import ThreadPoolExecutor#plusieur img en par

os.makedirs("data/images/train", exist_ok=True)
errors = []

def download(img_id):
    #get the info of the image with ther ID 
    info = coco.loadImgs(img_id)[0]
    path = f"data/images/train/{info['file_name']}"
    if os.path.exists(path):
        return
    try:
        #télécharge l'image
        r = requests.get(info["coco_url"], timeout=30)
        r.raise_for_status()
        #save img
        with open(path, "wb") as f:
            f.write(r.content)
    except Exception as e:
        errors.append(img_id)

wanted = kb_ids + val_ids
with ThreadPoolExecutor(max_workers=16) as ex:
    list(tqdm(ex.map(download, wanted), total=len(wanted)))

print("Images présentes :", len(os.listdir("data/images/train")))   # doit afficher 10100
print("Erreurs :", len(errors))

100%|██████████| 10100/10100 [20:22<00:00,  8.26it/s]

Images présentes : 10086
Erreurs : 14


In [4]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders:")
print(os.listdir())

Current working directory:
c:\Users\haman\Downloads\Telegram Desktop\image_pro\notebook

Files and folders:
['data', 'data_exploration.ipynb', 'embeddings.ipynb', 'one.ipynb', 'preprocessing_captions.ipynb', 'preprocessing_images.ipynb', 'splits_audit.ipynb']


In [5]:
import os

print("Contents of notebook/data:")
print(os.listdir("data"))

if os.path.exists("data/annotations"):
    print("\nContents of data/annotations:")
    print(os.listdir("data/annotations"))
else:
    print("\nThe annotations folder does not exist.")

Contents of notebook/data:
['annotations.zip', 'processed']

The annotations folder does not exist.
